In [ ]:
# Source: https://github.com/ros-tracing/tracetools_analysis/blob/caa8c05158bb2f898e1632188e57eb58c5e88d05/tracetools_analysis/analysis/callback_duration.ipynb

trace_dir = "/trace/2026-02-12_11-28-36"
print(f'Tracing data located at: {trace_dir}')

In [ ]:
import sys
# Add paths to tracetools_analysis and tracetools_read.
# There are two options:
#   1. from source, assuming a workspace with:
#       src/tracetools_analysis/
#       src/ros2/ros2_tracing/tracetools_read/
#sys.path.insert(0, '../')
#sys.path.insert(0, '../../../ros2/ros2_tracing/tracetools_read/')
#   2. from Debian packages, setting the right ROS 2 distro:
import os
ros_distro = os.environ.get("ROS_DISTRO")
sys.path.insert(0, f'/opt/ros/{ros_distro}/lib/python3.8/site-packages')
import datetime as dt

from bokeh.plotting import figure
from bokeh.plotting import output_notebook
from bokeh.io import show
from bokeh.layouts import row
from bokeh.models import ColumnDataSource
from bokeh.models import DatetimeTickFormatter
from bokeh.models import PrintfTickFormatter
from bokeh.palettes import Category10
import numpy as np
import pandas as pd

from tracetools_analysis.loading import load_file
from tracetools_analysis.processor.ros2 import Ros2Handler
from tracetools_analysis.utils.ros2 import Ros2DataModelUtil

In [ ]:
# Process
events = load_file(trace_dir)
handler = Ros2Handler.process(events)
#handler.data.print_data()

In [ ]:
data_util = Ros2DataModelUtil(handler.data)

callback_symbols = data_util.get_callback_symbols()

output_notebook()
psize = 450
# If the trace contains more callbacks, add colours here
# or use: https://docs.bokeh.org/en/3.2.2/docs/reference/palettes.html
colours = Category10[10]


### Analyze Callback Durations

In [ ]:
# Plot durations separately
colour_i = 0
for obj, symbol in callback_symbols.items():
    owner_info = data_util.get_callback_owner_info(obj)
    if owner_info is None:
        owner_info = '[unknown]'

    # Filter out internal subscriptions
    if '/parameter_events' in owner_info:
        continue

    # Duration
    duration_df = data_util.get_callback_durations(obj)
    starttime = duration_df.loc[:, 'timestamp'].iloc[0].strftime('%Y-%m-%d %H:%M')
    source = ColumnDataSource(duration_df)
    duration = figure(
        title=owner_info.replace(',', ',\n'),
        x_axis_label=f'start ({starttime})',
        y_axis_label='duration (ms)',
        width=psize, height=psize,
    )
    duration.title.align = 'left'
    duration.line(
        x='timestamp',
        y='duration',
        legend_label=str(symbol)[:60],
        line_width=2,
        source=source,
        line_color=colours[colour_i],
    )
    duration.legend.label_text_font_size = '11px'
    duration.xaxis[0].formatter = DatetimeTickFormatter(seconds='%Ss')

    # Histogram
    # (convert to milliseconds)
    dur_hist, edges = np.histogram(duration_df['duration'] * 1000 / np.timedelta64(1, 's'))
    duration_hist = pd.DataFrame({
        'duration': dur_hist, 
        'left': edges[:-1], 
        'right': edges[1:],
    })
    hist = figure(
        title='Duration histogram',
        x_axis_label='duration (ms)',
        y_axis_label='frequency',
        width=psize, height=psize,
    )
    hist.title.align = 'center'
    hist.quad(
        bottom=0,
        top=duration_hist['duration'], 
        left=duration_hist['left'],
        right=duration_hist['right'],
        fill_color=colours[colour_i],
        line_color=colours[colour_i],
    )

    colour_i += 1
    colour_i %= len(colours)
    show(row(duration, hist))

In [ ]:
# Plot durations in one plot
earliest_date = None
for obj, symbol in callback_symbols.items():
    duration_df = data_util.get_callback_durations(obj)
    thedate = duration_df.loc[:, 'timestamp'].iloc[0]
    if earliest_date is None or thedate <= earliest_date:
        earliest_date = thedate

starttime = earliest_date.strftime('%Y-%m-%d %H:%M')
duration = figure(
    title='Callback durations',
    x_axis_label=f'start ({starttime})',
    y_axis_label='duration (ms)',
    width=psize, height=psize,
)

colour_i = 0
for obj, symbol in callback_symbols.items():
    # Filter out internal subscriptions
    owner_info = data_util.get_callback_owner_info(obj)
    if not owner_info or '/parameter_events' in owner_info:
        continue

    duration_df = data_util.get_callback_durations(obj)
    source = ColumnDataSource(duration_df)
    duration.title.align = 'center'
    duration.line(
        x='timestamp',
        y='duration',
        legend_label=str(symbol)[:60],
        line_width=2,
        source=source,
        line_color=colours[colour_i],
    )
    colour_i += 1
    colour_i %= len(colours)
    duration.legend.label_text_font_size = '11px'
    duration.xaxis[0].formatter = DatetimeTickFormatter(seconds='%Ss')

show(duration)